# INF6083 - Projet P1
## Analyse du dataset Amazon Reviews 2023 - Books
### Équipe 7


# Tâche 0 - Chargement et échantillonnage des données
## 0.1 Importation des bibliothèques


In [ ]:
import pandas as pd
import numpy as np
import polars as pl
import duckdb
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import random
import gzip
import json
import matplotlib.pyplot as plt
import scipy.sparse as sp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
import networkx as nx
import time


## 0.2 Chargement des données (JSONL → Parquet)
Transformation du dataset en base Parquet pour un chargement rapide (10–50×). Fichier source : `data/Books.jsonl`.

In [ ]:
# Conversion JSONL → Parquet (DuckDB + Polars)
pl.scan_ndjson("data/Books.jsonl").sink_parquet("data/Books-polars.parquet")

con = duckdb.connect()
print("Converting JSONL → Parquet (this takes ~3-5 min)...")
con.execute("""
    COPY (
        SELECT *
        FROM read_json_auto(
            'data/Books.jsonl',
            format='newline_delimited',
            maximum_object_size=10485760
        )
    ) TO 'data/Books.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 1000000)
""")
print("Done!")
con.close()

## 0.3 Échantillonnage stratégique
- Filtrage des utilisateurs actifs (≥ 20 reviews)
- Sélection aléatoire de 50,000 utilisateurs
- Conservation de toutes leurs interactions


In [ ]:
# Échantillonnage stratifié (seed 42) — adapter RESOURCE_LIMITS à votre machine
DATA_PATH = "data/Books.parquet"
MIN_REVIEWS = 20
NUM_USERS = 50_000
SEED = 42

cluster = LocalCluster(
    n_workers=4,
    threads_per_worker=6,
    memory_limit="12GB",
)
client = Client(cluster)

# Reviews par utilisateur
print("Counting reviews per user...")
ddf = dd.read_parquet(DATA_PATH, columns=["user_id"])
user_counts = ddf.groupby("user_id").size().compute()
active_users = user_counts[user_counts >= MIN_REVIEWS].index.tolist()
print(f"Active users: {len(active_users):,}")
del ddf, user_counts

# Échantillon d'utilisateurs
random.seed(SEED)
sampled_users = set(random.sample(active_users, min(NUM_USERS, len(active_users))))
del active_users

# Filtrage et écriture
print("Filtering reviews...")
NEEDED_COLS = ["user_id", "parent_asin", "rating", "timestamp",
               "title", "text", "helpful_vote", "verified_purchase", "asin"]
ddf_full = dd.read_parquet(DATA_PATH, columns=NEEDED_COLS)
sampled_ddf = ddf_full[ddf_full["user_id"].isin(sampled_users)]
sampled_ddf.to_parquet(
    "samples/sampled_reviews-polars.parquet",
    write_index=False,
    overwrite=True,
)

result = dd.read_parquet("samples/sampled_reviews-polars.parquet")
print(f"Sampled: {len(result):,} reviews")

client.close()
cluster.close()

# Fichier unique + JSONL pour compatibilité
sampled_df = pd.read_parquet("samples/sampled_reviews-polars.parquet")
sampled_df.to_parquet("samples/sampled_reviews-polars-single-file.parquet")
sampled_df.to_json("samples/sampled_reviews-polars.jsonl", orient="records", lines=True)
print(f"Sampled: {len(sampled_df):,} reviews from {sampled_df['user_id'].nunique():,} users")

## 0.4 Analyse exploratoire
- Statistiques descriptives
- Distribution des ratings
- Sparsité
- Visualisations


In [ ]:
# TODO: Calcul des statistiques de base
pass

## 0.5 Prétraitement
- Nettoyage des ratings
- Filtrage utilisateurs/items
- Construction matrice utilisateur-item (CSR)
- Split train/test (80/20 stratifié)


In [ ]:
# TODO: Construction matrice sparse
pass

# Tâche 1 - Mesures de similarité
## 1.1 Implémentation des similarités
- Cosinus
- Pearson
- Jaccard


In [ ]:
# TODO: Implémenter similarité cosinus
# TODO: Implémenter similarité Pearson
# TODO: Implémenter similarité Jaccard
pass

## 1.2 Analyse comparative
- Sélection utilisateurs profils variés
- Top 10 voisins
- Heatmap
- Distribution des similarités


In [ ]:
# TODO: Analyse comparative des similarités
pass

# Tâche 2 - Représentation en graphe
## 2.1 Construction graphe biparti


In [ ]:
# TODO: Construire graphe biparti avec NetworkX
pass

## 2.2 Analyse du graphe
- Degré moyen
- Densité
- Centralité
- Clustering
- Composantes connexes


In [ ]:
# TODO: Calcul métriques graphe
pass

# Tâche 3 - Regroupement des utilisateurs
## 3.1 K-Means et détermination de K


In [ ]:
# TODO: Appliquer KMeans pour K = 3..8
pass

## 3.2 Analyse des clusters
- Taille
- Centres
- Moyennes
- Top livres
- Visualisation PCA / t-SNE


In [ ]:
# TODO: Analyse clusters + visualisation 2D
pass

# Tâche 4 - Prédiction des évaluations
## 4.1 Baselines


In [ ]:
# TODO: Baseline moyenne globale
# TODO: Baseline moyenne par livre
pass

## 4.2 k-NN collaboratif basé utilisateur


In [ ]:
# TODO: Implémentation k-NN
pass

## 4.3 Analyse des performances
- RMSE
- MAE
- Temps d'exécution


In [ ]:
# TODO: Tableau comparatif
pass

# Tâche 5 - Discussion et analyse critique
- Synthèse des résultats
- Limitations
- Défis de volumétrie
- Perspectives d'amélioration
